In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

In [3]:
# Random state seed
seed = 55

# Compile Data

In [4]:
DATA_DIR = Path("../DATA")
csv_files = sorted(DATA_DIR.glob('*.csv'))

data_list = []
y_tensor = None

for i, file in enumerate(csv_files):
    df = pd.read_csv(file)

    # Extract 288 features from end and convert to numpy
    only_features = df.iloc[:,-288:]
    features_np = only_features.to_numpy()

    data_list.append(features_np)

    # labels will be same for every file, so just extract from the first one
    if i == 0:
        labels = df.iloc[:,0]

        # If 'NSEP', label is 0
        # Otherwise, label is 1 (gt10, gt30, gt60, gt100)
        y_tensor = np.where(labels == 'NSEP', 0, 1)

X_tensor = np.stack(data_list, axis=-1)

In [5]:
counts = np.bincount(y_tensor)
print(f"Unique labels: {df.iloc[:,0].unique().tolist()}")
print(f"Total NSEP: {counts[0]}")
print(f"Total SEP:  {counts[1]}")
print()
print(f"X_tensor shape: {X_tensor.shape}")  
print(f"y_tensor shape: {y_tensor.shape}")

Unique labels: ['NSEP', 'gt100', 'gt10', 'gt60', 'gt30']
Total NSEP: 17625
Total SEP:  169

X_tensor shape: (17794, 288, 10)
y_tensor shape: (17794,)


# Train/Val/Test Split

In [6]:
# Separate 60% for training. Other 40% for val/test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_tensor, y_tensor,
    test_size=0.40,
    random_state=seed,
    stratify=y_tensor
)

# Separate val/test in half
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=seed,
    stratify=y_temp
)

print(f"Train set shape: {X_train.shape} | SEP count: {np.sum(y_train == 1)}")
print(f"Val set shape:   {X_val.shape}  | SEP count: {np.sum(y_val == 1)}")
print(f"Test set shape:  {X_test.shape}  | SEP count: {np.sum(y_test == 1)}")

Train set shape: (10676, 288, 10) | SEP count: 101
Val set shape:   (3559, 288, 10)  | SEP count: 34
Test set shape:  (3559, 288, 10)  | SEP count: 34


# Standard Normalization

In [7]:
# Reshape data to 2D for normalization
n_cases_train, n_time_train, n_features_train = X_train.shape
n_cases_val, n_time_val, n_features_val = X_val.shape 
n_cases_test, n_time_test, n_features_test = X_test.shape 

X_train_reshaped = X_train.reshape(-1, n_features_train)
X_val_reshaped = X_val.reshape(-1, n_features_val)
X_test_reshaped = X_test.reshape(-1, n_features_test)

# Apply standard scaling
scaler = StandardScaler()
X_train_reshaped = scaler.fit_transform(X_train_reshaped) 
X_val_reshaped = scaler.transform(X_val_reshaped) 
X_test_reshaped = scaler.transform(X_test_reshaped) 

# Reshape back to the original 3D shape
X_train = X_train_reshaped.reshape(n_cases_train, n_time_train, n_features_train)
X_val = X_val_reshaped.reshape(n_cases_val, n_time_val, n_features_val)
X_test = X_test_reshaped.reshape(n_cases_test, n_time_test, n_features_test)

print("Normalized X_train shape:", X_train.shape) 
print("Normalized X_val shape:  ", X_val.shape)   
print("Normalized X_test shape: ", X_test.shape)  
print()
print("X_Train Mean:", np.mean(X_train, axis=(0, 1)))
print("X_Train Std per feature: ", np.std(X_train, axis=(0, 1)))


Normalized X_train shape: (10676, 288, 10)
Normalized X_val shape:   (3559, 288, 10)
Normalized X_test shape:  (3559, 288, 10)

X_Train Mean: [-1.46528872e-14 -1.96920236e-16  2.48972383e-13  1.20160104e-13
  1.53093461e-12 -4.43188737e-16 -3.69680795e-15 -2.20859944e-15
  2.18222656e-13 -9.38713761e-15]
X_Train Std per feature:  [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


# Misc Classifications

In [8]:
# Flatten 
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_val_flat = X_val.reshape(X_val.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

In [ ]:
### Evaluation Metric Helper Functions ###

def true_skill_statistic(confusion_matrix):
    tn, fp, fn, tp = confusion_matrix.ravel()

    # Guard against divide by zero
    if (tp + fn == 0 or
        tn + fp == 0):
        return 0.0

    tss = tp / (tp + fn) - fp / (tn + fp)
    
    return float(tss)

def heidke_skill_score(confusion_matrix):
    tn, fp, fn, tp = confusion_matrix.ravel()

    numerator = 2 * ((tp * tn) - (fp * fn))
    denominator = ((tp + fn) * (fn + tn)) + ((tn + fp) * (tp + fp))

    # Guard against divide by zero
    if denominator == 0:
        return 0.0
        
    return float(numerator / denominator)

def evaluate_model_metrics(y_actual, y_pred, y_prob, model_name="Model"):    
    conf_matrix = confusion_matrix(y_actual, y_pred)

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_actual, y_pred),
        "Precision": precision_score(y_actual, y_pred),
        "Recall": recall_score(y_actual, y_pred),
        "F1-Score": f1_score(y_actual, y_pred),
        "ROC-AUC": roc_auc_score(y_actual, y_prob),
        "PR-AUC": average_precision_score(y_actual, y_prob),
        "TSS": true_skill_statistic(conf_matrix),
        "HSS2": heidke_skill_score(conf_matrix),
        "Confusion Matrix": conf_matrix
    }
    
    return metrics

### Logistic Regression

In [10]:
logreg_model = LogisticRegression(
    class_weight='balanced', 
    max_iter=1000, 
    random_state=seed
)

logreg_model.fit(X_train_flat, y_train)

y_val_pred = logreg_model.predict(X_val_flat)
y_val_prob = logreg_model.predict_proba(X_val_flat)[:, 1]

logreg_results = evaluate_model_metrics(
    y_actual=y_val, 
    y_pred=y_val_pred, 
    y_prob=y_val_prob, 
    model_name="Logistic Regression"
)
logreg_results

{'Model': 'Logistic Regression',
 'Accuracy': 0.9527957291373982,
 'Precision': 0.0410958904109589,
 'Recall': 0.17647058823529413,
 'F1-Score': 0.06666666666666667,
 'ROC-AUC': 0.5397329995828118,
 'PR-AUC': 0.034449073301951176,
 'TSS': 0.13675427617855654,
 'HSS2': 0.05197465640915058,
 'Confusion Matrix': array([[3385,  140],
        [  28,    6]])}

In [ ]:
xgb_model = XGBClassifier(
    scale_pos_weight=104,   # Weights class imabalance
    max_depth=3,            # Decision tree max depth
    n_estimators=500,       # No. of sequential decision trees
    learning_rate=0.05,     
    random_state=seed,
)

xgb_model.fit(X_train_flat, y_train)

y_val_pred_xgb = xgb_model.predict(X_val_flat)
y_val_prob_xgb = xgb_model.predict_proba(X_val_flat)[:, 1]

xgb_results = evaluate_model_metrics(
    y_actual=y_val, 
    y_pred=y_val_pred_xgb, 
    y_prob=y_val_prob_xgb, 
    model_name="XGBoost"
)
xgb_results

{'Model': 'XGBoost Baseline',
 'Accuracy': 0.9887608878898567,
 'Precision': 0.2,
 'Recall': 0.058823529411764705,
 'F1-Score': 0.09090909090909091,
 'ROC-AUC': 0.7741677096370463,
 'PR-AUC': 0.08776088005876845,
 'TSS': 0.056554025865665415,
 'HSS2': 0.08694425203314605,
 'Confusion Matrix': array([[3517,    8],
        [  32,    2]])}